# Week 11: Fine-Tuning Representation Models

In Week 5 we built a `TextClassifier` from random weights and trained it on 3,200 Amazon reviews. It reached about 80% accuracy, and it was confidently wrong on one review we kept coming back to:

> "not bad at all, actually pretty good"

Week 7 tried a different route: no training at all, just prompting a 1.5B-parameter generative model for every prediction. This week we go back to Week 5's setup, same data, same split, same metrics, and change one thing: instead of starting from random weights, we start from **BERT**, an encoder that has already read about 3.3 billion words.

**Runtime:** Colab, T4 GPU (Runtime > Change runtime type). Part A also runs on a CPU, slowly. Part B needs the GPU.

# A: Using a Pretrained Encoder As It Is

## Setup

The data cell below is Week 5's recipe: the same Amazon review CSV, the same balanced 2,000 positive + 2,000 negative subsample, the same seed, and the last 800 reviews held out for testing. Keeping the split fixed is what makes "beat 80%" a fair comparison. The loading is wrapped in `load_labeled_csv()` so you can point the whole notebook at your own data later.

We load `bert-base-uncased` with Hugging Face's `AutoModel`. This gives the encoder only, no task head. "uncased" means the tokenizer lowercases everything first.

In [ ]:
import csv, io, random, time, urllib.request
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)
random.seed(0)
torch.manual_seed(0)

def load_labeled_csv(source, text_col, label_col):
    # any CSV with a text column and an integer label column, from a URL or a local file
    if source.startswith("http"):
        raw = urllib.request.urlopen(source, timeout=30).read().decode("utf-8-sig")
    else:
        raw = open(source, encoding="utf-8-sig").read()
    rows = list(csv.DictReader(io.StringIO(raw)))
    return [r[text_col] for r in rows], [int(r[label_col]) for r in rows]

# Week 5 data, same recipe
url = "https://raw.githubusercontent.com/pycaret/pycaret/master/datasets/amazon.csv"
all_texts, all_labels = load_labeled_csv(url, text_col="reviewText", label_col="Positive")
pos_reviews = [t for t, y in zip(all_texts, all_labels) if y == 1]
neg_reviews = [t for t, y in zip(all_texts, all_labels) if y == 0]
random.shuffle(pos_reviews)
random.shuffle(neg_reviews)
N_PER_CLASS = 2000
texts = pos_reviews[:N_PER_CLASS] + neg_reviews[:N_PER_CLASS]
labels = [1] * N_PER_CLASS + [0] * N_PER_CLASS
combined = list(zip(texts, labels))
random.shuffle(combined)
texts, labels = zip(*combined)

N_TEST = 800
train_texts, test_texts = list(texts[:-N_TEST]), list(texts[-N_TEST:])
train_labels, test_labels = list(labels[:-N_TEST]), list(labels[-N_TEST:])
print(f"train: {len(train_texts)}   test: {len(test_texts)}")

MODEL = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
bert = AutoModel.from_pretrained(MODEL).to(DEVICE).eval()
print(f"{MODEL}: {sum(p.numel() for p in bert.parameters()) / 1e6:.0f}M parameters")

REVIEW = "not bad at all, actually pretty good"

## What pretraining gave BERT

BERT is the same kind of model as our Week 5 `TextClassifier`: transformer encoder blocks with **bidirectional** attention (no causal mask), so every token can look at every other token. The difference is scale and training. BERT-base has 12 blocks, 12 heads per block, 768-dimensional vectors and about 110M parameters, and before anyone used it for sentiment it was trained on BooksCorpus and English Wikipedia with one task: **masked language modeling**. Hide 15% of the tokens and predict them from both sides, for example "the battery was not [MASK] at all, actually pretty good". To get those right, BERT had to learn what "not" does to the word after it, which is exactly what our Week 5 model never learned from 3,200 reviews.

## What fine-tuning is

**Fine-tuning** means taking a model that has already been trained on a large general dataset and training it a little more on a small labelled dataset for one task. The key word is *starting*: training begins from the pretrained weights, not from random numbers, and it uses a small learning rate so those weights are nudged rather than replaced.

| | General (pretrained) model | Fine-tuned model |
|---|---|---|
| Trained on | 3.3B words of unlabelled text | the same, then 3,200 labelled reviews |
| Knows | English in general: grammar, word meaning, what "not" does | the same, plus what counts as a positive or negative review |
| Outputs | a 768-dim vector per token, no answer to any task | "positive" or "negative" with probabilities |
| Weights | the original BERT | the same architecture, every weight nudged, plus a new head |
| Good at | nothing specific yet, a starting point for many tasks | one task; one copy per task |

The architecture does not change. What changes is the numbers inside it, and a small task head on top.

One thing fine-tuning is **not**: a way to load facts into a model. It teaches a *behavior* (a task, a label set, an output format). If you need a model to answer from your documents or data, retrieval (Week 9) supplies the facts; fine-tuning can teach the model how to use them (Week 12).

## Three ways to classify the running review

We have now seen three ways to label "not bad at all, actually pretty good". They are worth putting side by side, because they are the three options you have for any classification project.

| | Week 5: train from scratch | Week 7: prompt Qwen | Week 11: fine-tune BERT |
|---|---|---|---|
| Starts from | random weights | pretrained, instruction-tuned LLM | pretrained encoder |
| Size | small, all learned from our data | 1.5B parameters | 110M parameters |
| Labelled data needed | 3,200 reviews | none (a few examples go in the prompt) | the same 3,200 reviews |
| What it reads per review | the review | instructions + examples + the review | the review |
| How it answers | head on the pooled vector | generates the next token, masked to the two label words | head on the pooled vector |
| Knows "not bad" is positive? | no: only ever saw "bad" in its 3,200 reviews | from its pretraining, if the prompt works | from pretraining, then sharpened by our labels |

How BERT helps our Week 5 classifier: it is the **same design**, an encoder, pooling and a head, so nothing about how we predict or evaluate changes. The only difference is that the encoder arrives already knowing English. Week 5 had to learn the meaning of every word from 3,200 short reviews; BERT starts with meanings learned from 3.3 billion words, so those 3,200 reviews only have to teach the task.

How it differs from Qwen: Qwen is about 14 times bigger, reads a long prompt for every review, and answers by *generating* text that we then force into one of two labels. It needs no training, which makes it the fastest way to get started. Fine-tuned BERT needs labelled data and a training run, but then it is small, reads only the review, and outputs the two probabilities directly. For a task you run thousands of times with a fixed set of labels, that is usually the better trade.

The tokenizer is **WordPiece**, a close cousin of the BPE we built in Week 2: common words stay whole, rare words split into pieces marked `##`. It also adds two special tokens: `[CLS]` at the start and `[SEP]` at the end. Let's see what BERT actually receives for our review.

In [ ]:
enc = tokenizer(REVIEW, return_tensors="pt")
print("tokens:", tokenizer.convert_ids_to_tokens(enc["input_ids"][0]))
print("ids   :", enc["input_ids"][0].tolist())
print("mask  :", enc["attention_mask"][0].tolist())

with torch.no_grad():
    out = bert(**enc.to(DEVICE))
print("last_hidden_state:", tuple(out.last_hidden_state.shape), " (batch, tokens, 768)")

One 768-dimensional vector per token, each one already shaped by every other token in the sentence. Notice there are no `##` pieces here because every word in this review is common. Try a rarer word like "headphones" or "unboxing" to see WordPiece split it.

## Turning BERT into features

To classify, we need one vector per review. We use the **masked mean pooling** from Week 5: average the token vectors, skipping padding positions (mask = 0). BERT also offers the `[CLS]` vector for this job; mean pooling usually works as well or better, and it keeps the link to Week 5.

We run all 4,000 reviews through BERT once, with `torch.no_grad()`, because nothing in BERT is going to change. This is **feature extraction**: BERT is frozen, it only turns text into vectors. It is exactly what we did with MiniLM in Week 6 and Week 9.

In [ ]:
MAX_LEN = 128   # covers nearly every review in this dataset

def mean_pool(hidden, mask):
    # Week 5 masked mean pooling: sum the real tokens, divide by how many there are
    mask = mask.unsqueeze(-1).float()
    return (hidden * mask).sum(dim=1) / mask.sum(dim=1)

@torch.no_grad()
def embed(texts, batch_size=64):
    vecs = []
    for i in range(0, len(texts), batch_size):
        enc = tokenizer(texts[i:i + batch_size], padding=True, truncation=True,
                        max_length=MAX_LEN, return_tensors="pt").to(DEVICE)
        hidden = bert(**enc).last_hidden_state
        vecs.append(mean_pool(hidden, enc["attention_mask"]).cpu())
    return torch.cat(vecs).numpy()

t0 = time.time()
X_train, X_test = embed(train_texts), embed(test_texts)
print(f"X_train {X_train.shape}   X_test {X_test.shape}   ({time.time() - t0:.0f}s)")

## Baseline 1: frozen BERT + a linear head

On top of the frozen vectors we train the simplest possible head: one linear layer with a softmax, trained with cross-entropy. scikit-learn's `LogisticRegression` is exactly that, already tuned to converge, so we use it here instead of writing the loop. It learns $768 \times 2 + 2 = 1{,}538$ numbers. BERT's 110M numbers stay untouched.

We report the same metrics as Week 5 and run the same five test reviews in Week 5's output format.

In [ ]:
head = LogisticRegression(max_iter=2000).fit(X_train, train_labels)
frozen_acc = accuracy_score(test_labels, head.predict(X_test))
print(f"frozen BERT + linear head: {frozen_acc:.1%}   (Week 5 from scratch: about 80%)\n")
print(classification_report(test_labels, head.predict(X_test),
                            target_names=["negative", "positive"], digits=3))

EXAMPLES = [
    "not really great",
    "This phone completely changed how I take photos, I love it",
    "Terrible battery life, would not buy again",
    "It's fine I guess, does what it says",
    "not bad at all, actually pretty good",
]

def predict_frozen(text):
    probs = head.predict_proba(embed([text]))[0]
    label = "positive" if probs[1] > probs[0] else "negative"
    return label, probs.tolist()

for review in EXAMPLES:
    label, probs = predict_frozen(review)
    print(f"{review!r:62s} -> {label:8s} (neg={probs[0]:.2f}, pos={probs[1]:.2f})")

Pretrained features alone already beat a model trained from scratch, with a head that has fewer parameters than Week 5's embedding table. But look closely at "not bad at all, actually pretty good" and the negation reviews: is the model sure, or just leaning? The head can only draw one straight line through the frozen vectors. If BERT's frozen vectors put "not bad" close to "bad", no line can fix that without hurting other reviews. The board work below shows why.

## Why the head alone is not enough

Every training step turns the model's mistake into an error signal. This week's question is **where that signal is allowed to go**: only into the head (frozen BERT), or further back into BERT itself (fine-tuning). A toy with two reviews shows the difference.

**Step 1. Setup.** Made-up numbers, 2 per vector instead of 768, every vector length 1.

| Review | Text | True label | BERT vector |
|---|---|---|---|
| A | "not bad at all, actually pretty good" | positive | $h_A = [0.8, 0.6]$ |
| T | "Terrible battery life" | negative | $h_T = [0.96, 0.28]$ |

For this toy only, read the first number as "negative evidence" and the second as "positive evidence". Real BERT's 768 numbers have no such simple meanings. A leans negative because it contains "bad".

The head has two rows, $w_{neg} = [1, 0]$ and $w_{pos} = [0, 1]$. Each class score (logit) is a dot product, $z = w \cdot h$.

**Step 2. Scores.**

Review A: $z_{neg} = 1(0.8) + 0(0.6) = 0.8$, $\quad z_{pos} = 0(0.8) + 1(0.6) = 0.6$. Since $0.8 > 0.6$, A is predicted negative: **wrong**.

Review T: $z_{neg} = 1(0.96) + 0(0.28) = 0.96$, $\quad z_{pos} = 0(0.96) + 1(0.28) = 0.28$. T is predicted negative: **correct**.

**Step 3. Probabilities and the error signal (Week 4 recap).**

A: $p_{pos} = \dfrac{1}{1 + e^{z_{neg} - z_{pos}}} = \dfrac{1}{1 + e^{0.2}} = \dfrac{1}{1 + 1.221} = 0.450$, so $p_{neg} = 0.550$. Loss $= -\ln 0.450 = 0.798$.

T: $p_{neg} = \dfrac{1}{1 + e^{z_{pos} - z_{neg}}} = \dfrac{1}{1 + e^{-0.68}} = \dfrac{1}{1 + 0.507} = 0.664$.

For softmax with cross-entropy, the error signal at the logits is $p - y$ (Week 4). A's true label is positive, so $y = [0, 1]$:

$p - y = [0.550, 0.450] - [0, 1] = [0.550, -0.550]$

The positive entry says "lower $z_{neg}$", the negative entry says "raise $z_{pos}$". **Now, where do we send this signal?**

**Option 1. Frozen BERT: the signal stops at the head.**

The gradient for the head is $\dfrac{\partial L}{\partial W} = (p - y)\, h_A^T$, so each row gets its own error times $h_A$:

- $w_{neg}$ row: $0.550 \times [0.8, 0.6] = [0.440, 0.330]$
- $w_{pos}$ row: $-0.550 \times [0.8, 0.6] = [-0.440, -0.330]$

Update with learning rate $\eta = 0.5$ (new = old $-\ \eta \times$ gradient):

$w_{neg}' = [1, 0] - 0.5 \cdot [0.440, 0.330] = [1 - 0.220,\ 0 - 0.165] = [0.780, -0.165]$

$w_{pos}' = [0, 1] - 0.5 \cdot [-0.440, -0.330] = [0 + 0.220,\ 1 + 0.165] = [0.220, 1.165]$

Check A (same vector, new head):

$z_{neg} = 0.780(0.8) + (-0.165)(0.6) = 0.624 - 0.099 = 0.525$, $\quad z_{pos} = 0.220(0.8) + 1.165(0.6) = 0.176 + 0.699 = 0.875$

$p_{pos} = \dfrac{1}{1 + e^{0.525 - 0.875}} = \dfrac{1}{1 + e^{-0.35}} = \dfrac{1}{1 + 0.705} = 0.587$. **A is fixed.**

Check T. The head is shared by every review, so T meets the new head too:

$z_{neg} = 0.780(0.96) + (-0.165)(0.28) = 0.749 - 0.046 = 0.703$, $\quad z_{pos} = 0.220(0.96) + 1.165(0.28) = 0.211 + 0.326 = 0.537$

$p_{neg} = \dfrac{1}{1 + e^{0.537 - 0.703}} = \dfrac{1}{1 + e^{-0.166}} = \dfrac{1}{1 + 0.847} = 0.541$

T dropped from 0.664 to 0.541. Fixing A moved the one shared line, and the line moved for every review.

| After one step | A: $p_{pos}$ | T: $p_{neg}$ |
|---|---|---|
| Before | 0.450 | 0.664 |
| Option 1: frozen, head moves | 0.587 | **0.541** |

So the frozen head can fix A, but only by moving the one line every review shares. What if the error signal could keep going, back into BERT, and move **A's vector** instead? That is fine-tuning, and it is where Class 2 starts.

# B: Fine-Tuning the Whole Model

## Letting the vector move

**Recap from Class 1.** Same toy: head rows $w_{neg} = [1, 0]$, $w_{pos} = [0, 1]$; review A ("not bad at all, actually pretty good", positive) has $h_A = [0.8, 0.6]$ and is predicted negative with $p_{pos} = 0.450$; review T ("Terrible battery life", negative) has $h_T = [0.96, 0.28]$ and $p_{neg} = 0.664$. A's error signal is $p - y = [0.550, -0.550]$. In Class 1 the signal stopped at the head: A was fixed ($p_{pos} = 0.587$), but T dropped to 0.541.

**Option 2. Fine-tuning: the signal continues back into the vector.**

The gradient for the vector is $\dfrac{\partial L}{\partial h} = W^T (p - y)$. With $W = \begin{bmatrix} 1 & 0 \\ 0 & 1 \end{bmatrix}$:

- first number: $1(0.550) + 0(-0.550) = 0.550$
- second number: $0(0.550) + 1(-0.550) = -0.550$

Update with $\eta = 0.5$:

$h_A' = [0.8, 0.6] - 0.5 \cdot [0.550, -0.550] = [0.8 - 0.275,\ 0.6 + 0.275] = [0.525, 0.875]$

The vector moved away from "negative" and toward "positive". Check A with the original head: $z_{neg} = 0.525$, $z_{pos} = 0.875$, so $p_{pos} = 0.587$. **A is fixed.** The head did not change and, in the toy, only A's vector moved, so T stays at $p_{neg} = 0.664$.

| After one step | A: $p_{pos}$ | T: $p_{neg}$ |
|---|---|---|
| Before | 0.450 | 0.664 |
| Option 1: frozen, head moves | 0.587 | **0.541** |
| Option 2: fine-tuned, vector moves | 0.587 | 0.664 |

Both options fix A by exactly the same amount on purpose: the head rows and $h_A$ all have length 1, so both updates change A's logits by exactly $\eta(p - y)$. The only difference is what happens to T.

**What the toy hides.**

| In the toy | In real fine-tuning |
|---|---|
| Option 1 uses backpropagation and Option 2 does not | Both use backpropagation. The difference is only where it stops: at the head, or all the way through BERT. |
| We edit $h_A$ directly | We never edit a vector. The gradient updates BERT's 110M weights, and the vector changes as a result. |
| T is untouched | BERT's weights are shared by every review, so T's vector moves a little too, and the head trains as well. |
| One step shows a side effect | Over many steps a frozen head finds the best line it can. Its real limit: if no single line separates the frozen vectors well, no amount of head training helps. Parts A and B test this on real data. |

The lasting difference: the head can only move one line for all reviews, while the encoder can move each review's vector according to the words inside it, so "bad" after "not" can land somewhere different from "bad" on its own. The rest of this class builds exactly that: a model where the gradient flows all the way through BERT.

## The model

`SentimentModel` is Week 5's `TextClassifier` with the from-scratch encoder swapped for pretrained BERT: encoder, masked mean pooling, dropout, a linear head. The head starts random; everything else starts from pretraining.

We load a **fresh copy** of BERT inside the model. Part A's frozen `bert` stays as it is, because at the end of class we compare the two.

In [ ]:
assert DEVICE == "cuda", "Part B trains BERT. Runtime > Change runtime type > T4 GPU"

class SentimentModel(nn.Module):
    def __init__(self, name=MODEL, n_classes=2):
        super().__init__()
        self.bert = AutoModel.from_pretrained(name)     # fresh pretrained copy
        self.drop = nn.Dropout(0.1)
        self.head = nn.Linear(self.bert.config.hidden_size, n_classes)   # random

    def forward(self, input_ids, attention_mask, **kwargs):
        hidden = self.bert(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        return self.head(self.drop(mean_pool(hidden, attention_mask)))

torch.manual_seed(0)
model = SentimentModel().to(DEVICE)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"trainable parameters: {trainable / 1e6:.1f}M   (Part A trained 1,538)")

## Batches and evaluation

Same tokenizer as Part A. We tokenize each batch on the fly and pad only to the longest review in that batch, which is faster than padding everything to 128. `evaluate()` returns test accuracy and the predictions.

Before training, the random head should be close to a coin flip. Checking that first means any number we see later comes from training, not luck.

In [ ]:
def batches(texts, labels, batch_size=32):
    # same loop as embed(): slice the next batch, tokenize it, return it with its labels
    for i in range(0, len(texts), batch_size):
        enc = tokenizer(texts[i:i + batch_size], padding=True, truncation=True,
                        max_length=MAX_LEN, return_tensors="pt").to(DEVICE)
        yield enc, torch.tensor(labels[i:i + batch_size], device=DEVICE)

@torch.no_grad()
def evaluate(model, texts=test_texts, labels=test_labels):
    model.eval()
    preds = []
    for enc, _ in batches(texts, labels, batch_size=64):
        preds += model(**enc).argmax(dim=-1).tolist()
    return accuracy_score(labels, preds), preds

acc, preds = evaluate(model)
print(f"untrained head: {acc:.1%} accuracy, predicts positive for {sum(preds)} of {len(preds)}")

## The training loop

The loop is Week 4's loop with three additions that matter for a big pretrained model:

The learning rate is **2e-5**, the range the BERT paper used (2e-5 to 5e-5). That is about 50 times smaller than the 1e-3 range that is normal when training from scratch, as in Weeks 4 and 5. Pretraining tuned BERT's weights very precisely, so fine-tuning should nudge them, not replace them. Steps that are too big erase what pretraining learned and the model falls back to predicting one class for everything; this is called **catastrophic forgetting**.

In [ ]:
from transformers import get_linear_schedule_with_warmup

def train(model, lr, epochs=2, batch_size=32):
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    for epoch in range(epochs):
        model.train()
        loss_sum, n, t0 = 0.0, 0, time.time()
        for enc, y in batches(train_texts, train_labels, batch_size):
            loss = F.cross_entropy(model(**enc), y)   # forward pass + loss
            opt.zero_grad()
            loss.backward()                            # gradients flow through the head AND all of BERT
            opt.step()
            loss_sum, n = loss_sum + loss.item() * len(y), n + len(y)
        acc, preds = evaluate(model)
        print(f"epoch {epoch + 1}: train loss {loss_sum / n:.3f}   test acc {acc:.1%}   "
              f"predicts positive for {sum(preds)}/{len(preds)}   ({time.time() - t0:.0f}s)")
    return model

random.seed(0)
model = train(model, lr=2e-5, epochs=2)

Two passes over the same 3,200 reviews Week 5 trained on for 6 epochs. Compare the final test accuracy with Week 5 (about 80%) and with frozen BERT in Part A.

## Results: Week 5's five reviews

The cell below prints the full report and the five test reviews side by side, frozen BERT from Part A against fine-tuned BERT. Watch the last one, "not bad at all, actually pretty good": frozen BERT called it negative with P(positive) = 0.00. This is Option 2 tested on the real model.

In [ ]:
acc, preds = evaluate(model)
print(f"fine-tuned BERT: {acc:.1%}   frozen BERT + head: {frozen_acc:.1%}   Week 5: about 80%\n")
print(classification_report(test_labels, preds, target_names=["negative", "positive"], digits=3))

@torch.no_grad()
def predict(text):
    model.eval()
    enc = tokenizer(text, return_tensors="pt").to(DEVICE)
    probs = F.softmax(model(**enc).float(), dim=-1)[0].tolist()
    return ("positive" if probs[1] > probs[0] else "negative"), probs

print(f"{'review':62s} {'frozen':>15s} {'fine-tuned':>15s}")
for review in EXAMPLES:
    f_label, f_probs = predict_frozen(review)
    t_label, t_probs = predict(review)
    print(f"{review!r:62s} {f_label:>8s} ({f_probs[1]:.2f}) {t_label:>8s} ({t_probs[1]:.2f})")
print("(number in brackets = P(positive))\n")

my_review = "Great, it broke on the second day."   # try your own
print("\nyour review ->", predict(my_review))

If the review flipped to positive, fine-tuning moved its representation the way the board work said it would. Nobody told the model about negation; the only signal was 3,200 labels, and it was enough because BERT already understood the sentence.